In [96]:
import pandas as pd
import numpy as np

# Creamos un diccionario con datos intencionalmente sucios
datos_sucios = {
    'ID_Poliza': ['A001', 'A002', 'A003', 'A004', 'A005', 'A006', 'A007', 'A008', 'A009', 'A010'],
    'Edad_Conductor': [25, 34, np.nan, 45, 150, 22, 55, np.nan, 30, 17], # Edades irreales y nulos
    'Zona_Geografica': ['Norte', 'SUR', 'norte', 'Este', ' SUR ', 'Oeste', 'nOrte', 'Este', 'Sur', 'Oeste'], # Problemas de formato
    'Prima_Mensual': ['$5000', '4500', '6000 ars', 'NaN', '5500', '$4200.50', '7000', 'ARS 4800', '5100', '3900'], # Texto mezclado con números
    'Siniestros_Previos': [0, 1, 0, 2, 0, 1, 0, 0, 3, 0]
}

# Lo convertimos a DataFrame y lo guardamos como CSV
df_falso = pd.DataFrame(datos_sucios)
df_falso.to_csv('polizas_desordenadas.csv', index=False)

print("¡Archivo 'polizas_desordenadas.csv' creado con éxito!")

¡Archivo 'polizas_desordenadas.csv' creado con éxito!


In [97]:
#1
df=pd.read_csv('polizas_desordenadas.csv')

df.head() # Muestra los valores de las primeras 5 filas
df.info() # Muestra info de nombres de columnas y tipos de datos
df.describe() # Muestra recuento de datos, promedio, desvío, min, cuartiles y max

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_Poliza           10 non-null     object 
 1   Edad_Conductor      8 non-null      float64
 2   Zona_Geografica     10 non-null     object 
 3   Prima_Mensual       9 non-null      object 
 4   Siniestros_Previos  10 non-null     int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 532.0+ bytes


,Edad_Conductor,Siniestros_Previos
count,8.000000,10.00000
mean,47.250000,0.70000
std,43.331777,1.05935
min,17.000000,0.00000
25%,24.250000,0.00000
50%,32.000000,0.00000
75%,47.500000,1.00000
max,150.000000,3.00000


In [98]:
#2.1
non_numeric_prima = df.Prima_Mensual.str.contains('[^0-9.-]', na=True) # El carácter ^ significa negación, pedimos todo lo que NO sea número, punto o guión
df.loc[non_numeric_prima].head() # .loc se utiliza para filtrar

,ID_Poliza,Edad_Conductor,Zona_Geografica,Prima_Mensual,Siniestros_Previos
0,A001,25.0,Norte,$5000,0
2,A003,NaN,norte,6000 ars,0
3,A004,45.0,Este,NaN,2
5,A006,22.0,Oeste,$4200.50,1
7,A008,NaN,Este,ARS 4800,0


In [99]:
#2.2
df['Prima_Mensual'] = df['Prima_Mensual'].str.replace('[^0-9.]','', regex=True) # regex es para decirle a Panda que no busque el texto literal [0-9.] sino que estamos usando el lenguaje de expresiones regulares
df['Prima_Mensual'] = pd.to_numeric(df['Prima_Mensual'], errors='coerce') # coerce lo usamos para decirle Python que si no puede convertir un valor en numero no se detenga y lo fuerce, convirtiendolo en nulo matemàtico real (NaN)
print(df['Prima_Mensual'])

0    5000.0
1    4500.0
2    6000.0
3       NaN
4    5500.0
5    4200.5
6    7000.0
7    4800.0
8    5100.0
9    3900.0
Name: Prima_Mensual, dtype: float64


In [100]:
#3
edad_conductor_promedio = df.loc[df['Edad_Conductor'].between(18,100), 'Edad_Conductor'].mean()  # .mean se usa para sacar el promedio
print(edad_conductor_promedio)

df['Edad_Conductor'] = df['Edad_Conductor'].fillna(edad_conductor_promedio) # fillna es para rellenar las celdas vacías
print(df['Edad_Conductor'])

df = df[df['Edad_Conductor'].between(18,100)]
print(df)

35.166666666666664
0     25.000000
1     34.000000
2     35.166667
3     45.000000
4    150.000000
5     22.000000
6     55.000000
7     35.166667
8     30.000000
9     17.000000
Name: Edad_Conductor, dtype: float64
  ID_Poliza  Edad_Conductor Zona_Geografica  Prima_Mensual  Siniestros_Previos
0      A001       25.000000           Norte         5000.0                   0
1      A002       34.000000             SUR         4500.0                   1
2      A003       35.166667           norte         6000.0                   0
3      A004       45.000000            Este            NaN                   2
5      A006       22.000000           Oeste         4200.5                   1
6      A007       55.000000           nOrte         7000.0                   0
7      A008       35.166667            Este         4800.0                   0
8      A009       30.000000             Sur         5100.0                   3


In [101]:
#4
set(df.Zona_Geografica) # set crea un conjunto, una estructura de datos desordenada, mutable y sin elementos duplicados
zona_geo_corregido = df['Zona_Geografica'].str.upper().str.strip() #upper convierte en mayusculas yu strip es para eliminar espacios en blanco al principio y final

df['Zona_Geografica'] = zona_geo_corregido
print(df)

  ID_Poliza  Edad_Conductor Zona_Geografica  Prima_Mensual  Siniestros_Previos
0      A001       25.000000           NORTE         5000.0                   0
1      A002       34.000000             SUR         4500.0                   1
2      A003       35.166667           NORTE         6000.0                   0
3      A004       45.000000            ESTE            NaN                   2
5      A006       22.000000           OESTE         4200.5                   1
6      A007       55.000000           NORTE         7000.0                   0
7      A008       35.166667            ESTE         4800.0                   0
8      A009       30.000000             SUR         5100.0                   3


In [102]:
#5
df1 = df.groupby('Zona_Geografica')['Prima_Mensual'].mean().reset_index() # groupby: Divide los datos según la zona, aplica el primedio a cada grupo, combina los resultados. reset_index: dado que groupby convierte en indice a Zona_Geografica, esto convierte en columna al indice
df1 = df1.rename(columns={'Prima_Mensual':'Prima_Promedio'})
print(df1)

  Zona_Geografica  Prima_Promedio
0            ESTE          4800.0
1           NORTE          6000.0
2           OESTE          4200.5
3             SUR          4800.0
